In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Preparation
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
)

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# 3. Model Architecture
class DigitClassifier(nn.Module):

    def __init__(self):
        super(DigitClassifier, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = DigitClassifier().to(DEVICE)

# 4. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


# 5. Training Loop
def train(model, loader, criterion, optimizer, device):
    model.train()
    for batch, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = criterion(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


# 6. Evaluation Loop
def evaluate(model, loader, criterion, device):
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += criterion(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= len(loader)
    accuracy = correct / len(loader.dataset)
    print(
        f"Test Error: \n Accuracy: {(accuracy*100):>0.1f}%, Avg loss: {test_loss:>8f}\n"
    )


# Run Training and Testing
for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train(model, train_loader, criterion, optimizer, DEVICE)
    evaluate(model, test_loader, criterion, DEVICE)
print("Done!")



100%|██████████| 9.91M/9.91M [00:00<00:00, 14.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 406kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.78MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]


Epoch 1
-------------------------------
Test Error: 
 Accuracy: 95.8%, Avg loss: 0.138530

Epoch 2
-------------------------------
Test Error: 
 Accuracy: 97.0%, Avg loss: 0.097508

Epoch 3
-------------------------------
Test Error: 
 Accuracy: 97.0%, Avg loss: 0.098169

Epoch 4
-------------------------------
Test Error: 
 Accuracy: 97.1%, Avg loss: 0.096435

Epoch 5
-------------------------------
Test Error: 
 Accuracy: 97.5%, Avg loss: 0.083914

Done!
